In [3]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 MB 31.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 25.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [torch]32m5/6 [torch]]x]


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Hyperparameters and synthetic data
batch_size   = 32
wide_dim     = 10      # Number of wide (dense) features
seq_length   = 20      # Length of sequence for Transformer
vocab_size   = 1000    # Token vocabulary size
embed_dim    = 32      # Embedding dimension for Transformer input
transformer_layers = 2
nhead        = 4
deep_fc_sizes = [64, 32]
wide_fc_size  = 16
final_fc_size = 1      # Binary classification

# Random toy data
# Wide features: continuous
X_wide = torch.randn(batch_size, wide_dim)
# Deep features: integer token IDs
X_seq  = torch.randint(0, vocab_size, (batch_size, seq_length))
# Binary targets
y      = torch.randint(0, 2, (batch_size, 1)).float()

# 2. Model definition
class WideAndDeepTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        # --- Deep (Transformer) part ---
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            activation='relu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)
        # After pooling the transformer's output, pass through MLP
        deep_layers = []
        in_dim = embed_dim
        for hid in deep_fc_sizes:
            deep_layers += [nn.Linear(in_dim, hid), nn.ReLU()]
            in_dim = hid
        self.deep_mlp = nn.Sequential(*deep_layers)
        
        # --- Wide part ---
        self.wide_fc = nn.Sequential(
            nn.Linear(wide_dim, wide_fc_size),
            nn.ReLU()
        )
        
        # --- Combined final layer ---
        combined_dim = deep_fc_sizes[-1] + wide_fc_size
        self.out_layer = nn.Linear(combined_dim, final_fc_size)
        
    def forward(self, x_wide, x_seq):
        # Wide branch
        wide_out = self.wide_fc(x_wide)
        
        # Deep branch
        # x_seq: (batch_size, seq_length) → embed → (seq_length, batch_size, embed_dim) for transformer
        emb = self.token_embed(x_seq)                                      # (B, S, E)
        emb = emb.permute(1, 0, 2)                                         # (S, B, E)
        trans_out = self.transformer(emb)                                  # (S, B, E)
        pooled   = trans_out.mean(dim=0)                                   # (B, E)
        deep_out = self.deep_mlp(pooled)                                   # (B, deep_fc_sizes[-1])
        
        # Combine
        combined = torch.cat([wide_out, deep_out], dim=1)                  # (B, combined_dim)
        logits   = self.out_layer(combined)                               # (B, 1)
        return logits

# 3. Instantiate, loss, optimizer
model = WideAndDeepTransformer()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 4. Training loop (toy example, just a few steps)
model.train()
for step in range(100):
    optimizer.zero_grad()
    logits = model(X_wide, X_seq)
    loss   = criterion(logits, y)
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        print(f"Step {step:03d} | Loss: {loss.item():.4f}")

# 5. Simple evaluation on the same data
model.eval()
with torch.no_grad():
    probs  = torch.sigmoid(model(X_wide, X_seq))
    preds  = (probs > 0.5).float()
    acc    = (preds == y).float().mean()
    print(f"Train Accuracy: {acc.item()*100:.2f}%")


/opt/anaconda3/envs/llm/lib/python3.13/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Step 000 | Loss: 0.6775
Step 010 | Loss: 0.6208
Step 020 | Loss: 0.4761
Step 030 | Loss: 0.2149
Step 040 | Loss: 0.0550
Step 050 | Loss: 0.0108
Step 060 | Loss: 0.0030
Step 070 | Loss: 0.0014
Step 080 | Loss: 0.0009
Step 090 | Loss: 0.0007
Train Accuracy: 100.00%
